-- Import nfl python package and load stats to test. ONLY 2026 CURRENTLY

In [16]:
import nflreadpy as nfl

stats = nfl.load_player_stats([2026]).to_pandas()
print(stats.shape)
print("Latest week:", stats["week"].max())
stats.columns.tolist()

(3407, 150)
Latest week: 4


['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'game_id',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_

-- load WR's stats and group all weeks of stats by player to total

In [17]:
wr = stats[(stats["position"] == "WR") & (stats["season_type"] == "REG")]

players = (
    wr.groupby(["player_id", "player_display_name"])
      .agg(
          games=("week", "nunique"),
          targets=("targets", "sum"),
          receptions=("receptions", "sum"),
          yards=("receiving_yards", "sum"),
          tds=("receiving_tds", "sum"),
          air_yards=("receiving_air_yards", "sum"),
          yac=("receiving_yards_after_catch", "sum"),
          epa=("receiving_epa", "sum"),
      )
      .reset_index()
)

players.sort_values("targets", ascending=False).head(15)

,player_id,player_display_name,games,targets,receptions,yards,tds,air_yards,yac,epa
59,00-0037239,Chris Olave,3,36,27,375,1,472,81,27.864391
81,00-0038543,Jaxon Smith-Njigba,3,36,27,405,6,348,121,28.975369
52,00-0036963,Amon-Ra St. Brown,3,35,23,228,5,275,73,10.125158
27,00-0035640,DK Metcalf,4,33,16,213,1,494,54,-0.537352
83,00-0038559,Michael Wilson,3,31,18,163,1,295,42,2.821888
152,00-0040667,Matthew Golden,3,30,15,253,1,416,97,6.670161
77,00-0038124,Christian Watson,3,29,17,284,4,304,104,5.102622
2,00-0031381,Davante Adams,3,29,18,358,2,498,106,15.096852
186,00-0041547,KC Concepcion,4,28,17,144,0,173,69,-1.010472
51,00-0036912,DeVonta Smith,3,27,19,235,1,323,39,1.293955


-- Set minimum targets for Sample Size
-- Get detailed WR stats 'rates'
-- Check # of WR's left with min targets and check head

In [18]:
MIN_TARGETS = 15  # adjust once you see the data

wr_rates = players[players["targets"] >= MIN_TARGETS].copy()
wr_rates["catch_rate"] = wr_rates["receptions"] / wr_rates["targets"]
wr_rates["yards_per_target"] = wr_rates["yards"] / wr_rates["targets"]
wr_rates["air_yards_per_target"] = wr_rates["air_yards"] / wr_rates["targets"]
wr_rates["yac_per_reception"] = wr_rates["yac"] / wr_rates["receptions"]
wr_rates["epa_per_target"] = wr_rates["epa"] / wr_rates["targets"]
wr_rates["targets_per_game"] = wr_rates["targets"] / wr_rates["games"]

print(wr_rates.shape)
wr_rates.sort_values("epa_per_target", ascending=False).head(15)


(52, 16)


,player_id,player_display_name,games,targets,receptions,yards,tds,air_yards,yac,epa,catch_rate,yards_per_target,air_yards_per_target,yac_per_reception,epa_per_target,targets_per_game
3,00-0031408,Mike Evans,3,16,12,137,2,175,17,22.387218,0.750000,8.562500,10.937500,1.416667,1.399201,5.333333
125,00-0039915,Ladd McConkey,3,15,12,183,1,208,23,16.918833,0.800000,12.200000,13.866667,1.916667,1.127922,5.000000
38,00-0036358,CeeDee Lamb,3,25,20,309,3,311,86,28.028233,0.800000,12.360000,12.440000,4.300000,1.121129,8.333333
36,00-0036322,Justin Jefferson,3,17,13,179,2,189,52,15.722178,0.764706,10.529412,11.117647,4.000000,0.924834,5.666667
7,00-0032464,Kalif Raymond,3,21,19,214,1,130,95,18.667351,0.904762,10.190476,6.190476,5.000000,0.888921,7.000000
58,00-0037238,Drake London,3,19,15,274,0,142,152,16.279734,0.789474,14.421053,7.473684,10.133333,0.856828,6.333333
81,00-0038543,Jaxon Smith-Njigba,3,36,27,405,6,348,121,28.975369,0.750000,11.250000,9.666667,4.481481,0.804871,12.000000
59,00-0037239,Chris Olave,3,36,27,375,1,472,81,27.864391,0.750000,10.416667,13.111111,3.000000,0.774011,12.000000
40,00-0036410,Tee Higgins,3,23,14,244,1,302,66,17.681941,0.608696,10.608696,13.130435,4.714286,0.768780,7.666667
165,00-0041037,Denzel Boston,4,22,13,284,2,383,109,15.270472,0.590909,12.909091,17.409091,8.384615,0.694112,5.500000


-- Detail WR comparison metrics

In [ ]:
features = [
    "targets_per_game",      # how involved he is
    "air_yards_per_target",  # how deep he's used
    "catch_rate",            # reliability
    "yards_per_target",      # efficiency
    "yac_per_reception",     # what he does after the catch
    "epa_per_target",        # overall value per target
]

wr_rates[features].describe()

,targets_per_game,air_yards_per_target,catch_rate,yards_per_target,yac_per_reception,epa_per_target
count,52.000000,52.000000,52.000000,52.000000,52.000000,52.000000
mean,7.190705,11.206595,0.634459,8.228909,3.975102,0.323473
std,1.834398,3.062451,0.113887,2.612210,2.071956,0.418983
min,5.000000,5.055556,0.352941,3.722222,0.400000,-0.559636
25%,5.666667,9.277097,0.582661,6.237500,2.500000,-0.007939
50%,7.000000,11.286096,0.628289,7.825758,3.750000,0.281760
75%,8.062500,13.309804,0.725000,10.115801,4.767857,0.575709
max,12.000000,17.409091,0.904762,14.421053,10.133333,1.399201


-- Standardize metrics with z-scores
-- Check data

In [20]:
z = (wr_rates[features] - wr_rates[features].mean()) / wr_rates[features].std()
z.index = wr_rates["player_display_name"].values
print(z)

                    targets_per_game  air_yards_per_target  catch_rate  \
Keenan Allen               -0.285673             -1.634833    0.136462   
Davante Adams               1.349741              1.948054   -0.120900   
Mike Evans                 -1.012524             -0.087869    1.014524   
Stefon Diggs                0.077752             -0.868958   -0.382392   
Kalif Raymond              -0.103961             -1.637943    2.373428   
Mack Hollins               -1.012524              0.299892    1.014524   
Courtland Sutton           -1.012524             -0.516448   -1.180629   
DJ Moore                   -0.649099              1.510796   -0.205006   
DK Metcalf                  0.577462              1.228788   -1.313669   
Terry McLaurin              0.077752              0.585611   -1.579748   
Justin Jefferson           -0.830811             -0.029045    1.143650   
CeeDee Lamb                 0.622890              0.402751    1.453554   
Tee Higgins                 0.259465  

-- Define comparison function

In [21]:
def most_similar(name, n=5):
    distances = ((z - z.loc[name]) ** 2).sum(axis=1) ** 0.5
    return distances.drop(name).sort_values().head(n)

# pick any name from the list
wr_rates["player_display_name"].tolist()

['Keenan Allen',
 'Davante Adams',
 'Mike Evans',
 'Stefon Diggs',
 'Kalif Raymond',
 'Mack Hollins',
 'Courtland Sutton',
 'DJ Moore',
 'DK Metcalf',
 'Terry McLaurin',
 'Justin Jefferson',
 'CeeDee Lamb',
 'Tee Higgins',
 'Jaylen Waddle',
 "Ja'Marr Chase",
 'DeVonta Smith',
 'Amon-Ra St. Brown',
 'Drake London',
 'Chris Olave',
 'Jameson Williams',
 'George Pickens',
 'Khalil Shakir',
 'Garrett Wilson',
 "Wan'Dale Robinson",
 'Christian Watson',
 'Dontayvion Wicks',
 'Jaxon Smith-Njigba',
 'Quentin Johnston',
 'Michael Wilson',
 'Tre Tucker',
 'Parker Washington',
 'Xavier Hutchinson',
 'Jordan Addison',
 'Josh Downs',
 'Rashee Rice',
 'Malik Nabers',
 'Ryan Flournoy',
 'Devaughn Vele',
 'Jalen Coker',
 'Roman Wilson',
 'Malik Washington',
 'Adonai Mitchell',
 'Xavier Worthy',
 'Ladd McConkey',
 'Tetairoa McMillan',
 'Emeka Egbuka',
 'Matthew Golden',
 'Tre Harris',
 'Luther Burden III',
 'Denzel Boston',
 'Carnell Tate',
 'KC Concepcion']

TESTER
TESTER

In [24]:
most_similar("Jaxon Smith-Njigba", n=10)

Chris Olave          1.372396
Garrett Wilson       2.359893
CeeDee Lamb          2.401291
Amon-Ra St. Brown    2.499224
Christian Watson     2.635380
Jalen Coker          2.800727
Ja'Marr Chase        2.845704
Tee Higgins          2.911925
DeVonta Smith        2.998386
Josh Downs           3.092949
dtype: float64

In [23]:
def weekly_view(name):
    cols = ["week", "targets", "receptions", "receiving_yards", "receiving_air_yards", "receiving_epa"]
    return wr[wr["player_display_name"] == name][cols].sort_values("week")

weekly_view("Davante Adams")

,week,targets,receptions,receiving_yards,receiving_air_yards,receiving_epa
23,1,6,3,26,73,-1.704729
1148,2,10,8,195,198,11.752949
2255,3,13,7,137,227,5.048632
